In [43]:
# 1. Object Identification
import pandas as pd

source1 = pd.read_csv("../01_data/source1_sample.csv")
source1.head()

,Object_ID,Validity_From,Validity_To,State,Object_Type
0,OBJ10000,20250311,20260630,Planning,Type_C
1,OBJ10000,20260701,20281231,Active,Type_C
2,OBJ10001,20240927,20250630,Planning,Type_A
3,OBJ10001,20250701,20260630,Active,Type_A
4,OBJ10002,20240312,20281231,Planning,Type_C


In [44]:
source1.shape

(1599, 5)

In [45]:
source2 = pd.read_csv("../01_data/source2_sample.csv")
source2.head()

,Object_ID,Object_Class,Validity_From,Validity_To
0,OBJ10000,Class_3,20261212,20271211
1,OBJ10001,Class_1,20261212,20271211
2,OBJ10002,Class_3,20261212,20271211
3,OBJ10003,Class_1,20261212,20271211
4,OBJ10004,Class_3,20261212,20271211


In [46]:
source2.shape

(800, 4)

In [47]:
source1["Object_ID"].nunique()

800

In [48]:
source2["Object_ID"].nunique()

800

In [49]:
source1["Object_ID"].value_counts().value_counts().sort_index()
# 799 Object_IDs appear exactly twice in source1 (Planning + Active), 1 Object_ID appears once (only Planning)

count
1      1
2    799
Name: count, dtype: int64

In [50]:
source2["Object_ID"].value_counts().value_counts().sort_index()
# no duplicates in source2 – each Object_ID appears exactly once

count
1    800
Name: count, dtype: int64

In [51]:
source1[source1.duplicated(subset=["Object_ID"], keep=False)] \
.sort_values(["Object_ID","Validity_From"])

,Object_ID,Validity_From,Validity_To,State,Object_Type
0,OBJ10000,20250311,20260630,Planning,Type_C
1,OBJ10000,20260701,20281231,Active,Type_C
2,OBJ10001,20240927,20250630,Planning,Type_A
3,OBJ10001,20250701,20260630,Active,Type_A
5,OBJ10003,20240121,20251231,Planning,Type_A
...,...,...,...,...,...
1594,OBJ10797,20261130,20270912,Active,Type_A
1595,OBJ10798,20240108,20250210,Planning,Type_A
1596,OBJ10798,20250211,20270116,Active,Type_A
1597,OBJ10799,20240212,20250918,Planning,Type_C


In [52]:
source1[source1.duplicated()]
# no "real" duplicates that share the same object_ID and Validity_From, Validity_To, State, and Object_Type meaning that the duplicates are a result of data versioning

,Object_ID,Validity_From,Validity_To,State,Object_Type


In [53]:
source2[source2.duplicated()]
# no "real" duplicates that share the same object_ID and Validity_From, Validity_To, State, and Object_Type but there is only one duplicate in that source anyways

,Object_ID,Object_Class,Validity_From,Validity_To


In [54]:
# now we check if every Object_ID in source1 is also in source2 and vice versa
set(source1["Object_ID"]) - set(source2["Object_ID"])

set()

In [55]:
set(source2["Object_ID"]) - set(source1["Object_ID"])

set()

In [56]:
# 2. Relevance Filtering
# we need to make sure only relevant objects of Source 1 exist in Source 2. Relevance is determined by temporal validity.
# source 2 validity is identical for all objects (versioning handled differently)
# NOTE: extract after datetime conversion to avoid datetime vs int comparison
# formatting to date
source1["Validity_From"] = pd.to_datetime(source1["Validity_From"], format="%Y%m%d")
source1["Validity_To"] = pd.to_datetime(source1["Validity_To"], format="%Y%m%d")

source2["Validity_From"] = pd.to_datetime(source2["Validity_From"], format="%Y%m%d")
source2["Validity_To"] = pd.to_datetime(source2["Validity_To"], format="%Y%m%d")

In [57]:
# extract validity boundaries from source2 (after datetime conversion!)
valid_from_s2 = source2["Validity_From"].iloc[0]
valid_to_s2 = source2["Validity_To"].iloc[0]
valid_from_s2, valid_to_s2

(Timestamp('2026-12-12 00:00:00'), Timestamp('2027-12-11 00:00:00'))

In [58]:
# Interval overlap check: a source 1 object is "relevant" if its validity period overlaps with the source 2 validity period.
#
# Overlap condition:  S1.From <= S2.To  AND  S1.To >= S2.From
#
#   Case 1 — NOT relevant (S1 ends before S2 starts):  S1:|------|           S2:          |--------|
#   Case 2 — NOT relevant (S1 starts after S2 ends):   S1:          |------| S2:|--------|
#   Case 3 — Relevant (partial overlap left):          S1:|----------|       S2:      |--------|
#   Case 4 — Relevant (partial overlap right):         S1:        |--------| S2:|--------|
#   Case 5 — Relevant (S1 inside S2):                  S1:    |--|           S2:|--------|
#   Case 6 — Relevant (S1 encloses S2):                S1:|----------------| S2:   |--------|
relevant_source1 = source1[
    (source1["Validity_From"] <= valid_to_s2) &
    (source1["Validity_To"] >= valid_from_s2)
].copy()

In [59]:
# complement: objects with NO temporal overlap (case 1 or 2)
list_a_not_relevant_source1 = source1[
    (source1["Validity_To"] < valid_from_s2) |
    (source1["Validity_From"] > valid_to_s2)
].copy()

In [60]:
len(source1)

1599

In [61]:
len(relevant_source1)

818

In [62]:
len(list_a_not_relevant_source1)
# 781 (not relevant) + 818 (relevant) = 1599 (full data set) --> every record of source1 was categorized as either "relevant" or "not relevant"

781

In [63]:
relevant_source1.head()

,Object_ID,Validity_From,Validity_To,State,Object_Type
1,OBJ10000,2026-07-01,2028-12-31,Active,Type_C
4,OBJ10002,2024-03-12,2028-12-31,Planning,Type_C
8,OBJ10004,2026-02-27,2027-04-17,Active,Type_C
10,OBJ10005,2026-01-16,2027-08-11,Active,Type_C
11,OBJ10006,2024-12-25,2027-05-07,Planning,Type_D


In [64]:
list_a_not_relevant_source1.head()

,Object_ID,Validity_From,Validity_To,State,Object_Type
0,OBJ10000,2025-03-11,2026-06-30,Planning,Type_C
2,OBJ10001,2024-09-27,2025-06-30,Planning,Type_A
3,OBJ10001,2025-07-01,2026-06-30,Active,Type_A
5,OBJ10003,2024-01-21,2025-12-31,Planning,Type_A
6,OBJ10003,2026-01-01,2026-06-30,Active,Type_A


In [65]:
# 3. Classification Mapping
type_mapping = {
    "Type_A": "Class_1",
    "Type_B": "Class_2",
    "Type_C": "Class_3"
}

In [66]:
relevant_source1["Mapped_Class"] = relevant_source1["Object_Type"].map(type_mapping)

In [67]:
relevant_source1.head()

,Object_ID,Validity_From,Validity_To,State,Object_Type,Mapped_Class
1,OBJ10000,2026-07-01,2028-12-31,Active,Type_C,Class_3
4,OBJ10002,2024-03-12,2028-12-31,Planning,Type_C,Class_3
8,OBJ10004,2026-02-27,2027-04-17,Active,Type_C,Class_3
10,OBJ10005,2026-01-16,2027-08-11,Active,Type_C,Class_3
11,OBJ10006,2024-12-25,2027-05-07,Planning,Type_D,NaN


In [68]:
relevant_source1["Mapped_Class"].isna().sum()
# 208 objects have no valid classification mapping

np.int64(208)

In [69]:
relevant_source1[relevant_source1["Mapped_Class"].isna()]

,Object_ID,Validity_From,Validity_To,State,Object_Type,Mapped_Class
11,OBJ10006,2024-12-25,2027-05-07,Planning,Type_D,NaN
12,OBJ10006,2027-05-08,2028-04-02,Active,Type_D,NaN
18,OBJ10009,2025-09-29,2027-07-20,Active,Type_D,NaN
26,OBJ10013,2025-10-22,2028-12-22,Active,Type_D,NaN
41,OBJ10021,2025-03-30,2026-12-17,Planning,Type_D,NaN
...,...,...,...,...,...,...
1544,OBJ10772,2025-12-29,2028-10-13,Active,Type_D,NaN
1546,OBJ10773,2026-01-15,2027-03-21,Active,Type_D,NaN
1567,OBJ10784,2025-04-09,2027-03-06,Planning,Type_D,NaN
1568,OBJ10784,2027-03-07,2029-08-22,Active,Type_D,NaN


In [70]:
merged_data = relevant_source1.merge(
    source2,
    on="Object_ID",
    how="left"
)

In [71]:
merged_data.head()

,Object_ID,Validity_From_x,Validity_To_x,State,Object_Type,Mapped_Class,Object_Class,Validity_From_y,Validity_To_y
0,OBJ10000,2026-07-01,2028-12-31,Active,Type_C,Class_3,Class_3,2026-12-12,2027-12-11
1,OBJ10002,2024-03-12,2028-12-31,Planning,Type_C,Class_3,Class_3,2026-12-12,2027-12-11
2,OBJ10004,2026-02-27,2027-04-17,Active,Type_C,Class_3,Class_3,2026-12-12,2027-12-11
3,OBJ10005,2026-01-16,2027-08-11,Active,Type_C,Class_3,Class_3,2026-12-12,2027-12-11
4,OBJ10006,2024-12-25,2027-05-07,Planning,Type_D,NaN,NaN,2026-12-12,2027-12-11


In [72]:
# 4. Temporal Alignment
# now we need to make sure we take different object states into account as only "active" objects of source1 have to exist in source2
active_source1 = relevant_source1[relevant_source1["State"] == "Active"].copy()

len(active_source1)

692

In [73]:
# these objects are active during the validity period Source 2 and therefore have to exist in Source 2
planning_source1 = relevant_source1[relevant_source1["State"] == "Planning"].copy()

In [74]:
planning_objects = set(planning_source1["Object_ID"])
active_objects = set(active_source1["Object_ID"])

In [75]:
missing_active = planning_objects - active_objects

len(missing_active)
# we found 1 object that has a planning state during the validity persiod of Source 2 but no following active state
# this indicates source data issues as this object does not seem to follow the expected state transition logic of Source 1.

1

In [76]:
# 5. Result Classification
# now we categorize our findings by creating lists
active_objects = active_source1["Object_ID"].unique()

In [77]:
# these objects need to exist in source 2
source2_relevant = source2[
    source2["Object_ID"].isin(active_objects)
].copy()

In [78]:
comparison = active_source1.merge(
    source2_relevant,
    on="Object_ID",
    how="left",
    suffixes=("_s1", "_s2")
)

In [79]:
# objects where Mapped_Class != Object_Class
class_mismatch = comparison[comparison["Mapped_Class"] != comparison["Object_Class"]]

In [80]:
list_b_consistent = comparison[
    comparison["Mapped_Class"] == comparison["Object_Class"]
].copy()

In [81]:
# note: list c contains all Type_D objects — Type_D has no mapping (Mapped_Class=NaN)
# and source 2 also has no Object_Class for these (NaN). NaN != NaN is True in pandas,
# so these are not real mismatches but consistently unclassified objects.
list_c_inconsistent = comparison[
    (comparison["Mapped_Class"] != comparison["Object_Class"]) |
    (comparison["Object_Class"].isna())
].copy()

In [82]:
list_d_source_issues = relevant_source1[
    relevant_source1["Object_ID"].isin(missing_active)
].copy()

len(list_d_source_issues)

1

In [83]:
print("List a - not relevant:", len(list_a_not_relevant_source1))
print("List b - consistent matches:", len(list_b_consistent))
print("List c - inconsistent matches:", len(list_c_inconsistent))
print("List d - source data issues:", len(list_d_source_issues))

List a - not relevant: 781
List b - consistent matches: 519
List c - inconsistent matches: 173
List d - source data issues: 1


In [84]:
outputs = {
    "list_a_not_relevant_source1": list_a_not_relevant_source1,
    "list_b_consistent": list_b_consistent,
    "list_c_inconsistent": list_c_inconsistent,
    "list_d_source_issues": list_d_source_issues
}

for name, df in outputs.items():
    df.to_csv(f"{name}.csv", index=False)

In [85]:
# remember we found 818 relevant records of source1 before? note that list b + list c + list d != 818 due to the fact that two records can be related to the same object.
# this is why we have to check for unique objects: the amount of unique objects should be eqaul to the sum of unique objects of list b + list c + list d
print("Relevant objects:", relevant_source1["Object_ID"].nunique())
print("Classified objects:",
      list_b_consistent["Object_ID"].nunique()
    + list_c_inconsistent["Object_ID"].nunique()
    + list_d_source_issues["Object_ID"].nunique())

Relevant objects: 693
Classified objects: 693
